In [ ]:
# bootstrap: raiz do projeto (sobe a arvore ate achar CLAUDE.md)
import os, pathlib
_r = pathlib.Path.cwd()
while not (_r / 'CLAUDE.md').exists() and _r != _r.parent:
    _r = _r.parent
os.chdir(_r)
print('raiz:', os.getcwd())

# Experimento - Modelo MENSAL (valor e quantidade)

Previsao do **total do mes** (valor R$ e quantidade de propostas), feita **antes do mes comecar**
(so com dados de meses anteriores). Complementa o `experimento_sarimax.ipynb`, que foca no diario
e na estrategia *nowcast* (total do mes atualizado dia a dia).

## Por que a estrutura dos dados e o problema central

O historico util e curto: os backups vao de ~2025-02 a 2026-04 (**~15 meses**). Isso condiciona tudo:

| | Serie temporal mensal (1 ponto/mes) | Linha por mes + lags (estilo `_vies`) |
|---|---|---|
| Sazonalidade anual (m=12) | **inviavel** (precisa 2+ ciclos = 24m+) | capturavel via `mes_sin/cos` |
| TPOT / AutoML | nao se aplica | **overfitta** ~15 linhas -> usar leve + CV |
| Nº de dias uteis do mes (19-23, driver dominante) | invisivel | **feature explicita** (deterministico, conhecido a frente) |

**Decisao:** representacao **linha-por-mes com lags** (supervisionada), com o alvo reformulado
para **media por dia util** (`total / nº_dias_uteis`). Isso remove a maior fonte de variacao
(calendario) e estabiliza o ajuste com N pequeno. A previsao do total e reconstruida como
`media_util_prevista x nº_dias_uteis_do_mes` (o nº de dias uteis e determinístico). Mantemos
tambem a **serie mensal pura** para o SARIMA.

Modelos avaliados (todos por **janela expansivel** mes a mes, sem vazamento):
- **Baselines**: naive (mes-1), sazonal (mes-12), media-util MM3 x dias uteis.
- **SARIMA** na serie mensal (sazonalidade anual so quando ha >= 24 meses).
- **Regressoes**: Ridge, Linear, RandomForest sobre a tabela supervisionada.
- **TPOT** (AutoML) leve -- fortemente *caveated* pelo N pequeno.

## 1 - Imports e configuracao

In [ ]:
import warnings, hashlib, pickle, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

import clickhouse_connect
from workadays import workdays as wd

from statsmodels.tsa.statespace.sarimax import SARIMAX
import pmdarima as pm

from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# TPOT e opcional (import pesado); so carrega se for usar
try:
    from tpot import TPOTRegressor
    _HAS_TPOT = True
except Exception as _e:
    _HAS_TPOT = False
    print('[TPOT indisponivel]', _e)

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.2f}'.format)
print('Imports OK | TPOT:', _HAS_TPOT)

In [ ]:
# ==================== CONFIGURACAO ====================
HOST = '10.101.150.150'; PORT = 8123
USER = 'debora_jesus';  PASS = '3fDsSF1$pe1yDv'

HIST_DIAS   = 540          # janela ampla; o DB limita ao que existir
MIN_TREINO_M = 6           # minimo de meses de historico antes de comecar a avaliar
COBERTURA_MIN = 0.80       # fracao dos dias uteis do mes que precisa ter dados p/ ser "completo"
os.makedirs('cache', exist_ok=True)

# Cobrir todo o periodo dos backups tambem (estende HIST_DIAS se preciso)
_mins = []
for _bp, _c in [('resultados/backup_valor.xlsx', 'Dia Referência'),
                ('resultados/backup_qtd.xlsx',   'Data previsao')]:
    try:
        _b = pd.read_excel(_bp); _d = pd.to_datetime(_b[_c], errors='coerce').dropna()
        if len(_d): _mins.append(_d.min())
    except Exception as _e:
        print('aviso ao ler backup', _bp, '->', _e)
if _mins:
    HIST_DIAS = max(HIST_DIAS, (datetime.today() - min(_mins)).days + 90)
print(f'HIST_DIAS={HIST_DIAS} | MIN_TREINO_M={MIN_TREINO_M}')

In [ ]:
# ==================== DADOS (cache 1x/dia) ====================
query = f"""
SELECT
    toStartOfFifteenMinutes(ultimaalteracao) AS Intervalo,
    SUM(valor) AS valor,
    COUNT(propostaid) AS qntd
FROM crefaz.ft_proposta fp
WHERE
    propostaetapaid = 16
    AND propostadecisaoid IS NULL
    AND toDate(ultimaalteracao) BETWEEN toDate(today() - INTERVAL {HIST_DIAS} DAY) AND toDate(today() - INTERVAL 1 DAY)
GROUP BY Intervalo
"""

_key   = hashlib.md5(f"{pd.Timestamp.today().date()}|{HIST_DIAS}|{query}".encode()).hexdigest()[:12]
_cache = os.path.join('cache', f'query_mensal_{_key}.pkl')
_ok = False
if os.path.exists(_cache):
    try:
        with open(_cache, 'rb') as _f:
            df = pickle.load(_f)
        _ok = True; print(f'[cache hit] <- {_cache}')
    except Exception as _e:
        print('[cache invalido]', _e)
if not _ok:
    _cli = clickhouse_connect.get_client(host=HOST, port=PORT, username=USER, password=PASS)
    df   = _cli.query_df(query)
    try:
        with open(_cache, 'wb') as _f:
            pickle.dump(df, _f)
        print(f'[cache saved] -> {_cache}')
    except Exception as _e:
        print('[cache erro]', _e)

df['Intervalo'] = df['Intervalo'].astype(str)
df['data'] = pd.to_datetime(pd.to_datetime(df['Intervalo'].str.split('+').str[0]).dt.date)
print(f'Propostas: {len(df)} registros 15min | {df["data"].min().date()} -> {df["data"].max().date()}')

In [ ]:
# Frame diario agregado (1 linha por dia)
_dd = df.groupby('data').agg(qntd=('qntd', 'sum'), valor=('valor', 'sum')).reset_index()
_dd['qntd']  = pd.to_numeric(_dd['qntd'],  errors='coerce').astype(float)
_dd['valor'] = pd.to_numeric(_dd['valor'], errors='coerce').astype(float)
# so dias uteis (seg-sex, sem feriado BR) -- fins de semana/feriados sao ~0 e nao contam mes util
_hol = _dd['data'].apply(lambda x: bool(wd.is_holiday(x.date(), country='BR')))
df_diario = _dd[(_dd['data'].dt.dayofweek < 5) & (~_hol)].sort_values('data').reset_index(drop=True)
print(f'df_diario (dias uteis): {len(df_diario)} dias | '
      f'{df_diario["data"].min().date()} -> {df_diario["data"].max().date()}')

## 2 - Estrutura dos dados: da serie diaria para a tabela mensal

- `n_dias_uteis_mes(ano, mes)`: quantos dias uteis (seg-sex, sem feriado BR) o mes tem -- **deterministico**,
  logo conhecido antes do mes comecar. E o driver dominante do total mensal.
- **Mes completo**: mantemos so meses cujo periodo de dados cobre >= `COBERTURA_MIN` dos dias uteis
  (descarta o 1º mes parcial e o mes corrente em andamento).
- **`tab_mensal`**: 1 linha por mes com `tot_valor`, `tot_qntd`, `n_uteis` e as **medias por dia util**
  (`media_valor = tot_valor / n_uteis`, idem qtd). As medias sao o alvo estavel; o total e reconstruido
  multiplicando pela contagem de dias uteis.

In [ ]:
def n_dias_uteis_mes(ano, mes):
    """Dias uteis (seg-sex, sem feriado BR) no mes-calendario inteiro."""
    _ini = pd.Timestamp(ano, mes, 1)
    _fim = _ini + pd.offsets.MonthEnd(0)
    _rng = pd.date_range(_ini, _fim, freq='D')
    return int(sum((x.weekday() < 5) and not wd.is_holiday(x.date(), country='BR') for x in _rng))

# agrega o diario por mes-calendario
_g = df_diario.copy()
_g['ano'] = _g['data'].dt.year
_g['mes'] = _g['data'].dt.month
_agg = _g.groupby(['ano', 'mes']).agg(
    tot_valor=('valor', 'sum'), tot_qntd=('qntd', 'sum'),
    dias_com_dados=('data', 'nunique'),
    d_ini=('data', 'min'), d_fim=('data', 'max')).reset_index()
_agg['n_uteis']    = _agg.apply(lambda r: n_dias_uteis_mes(int(r['ano']), int(r['mes'])), axis=1)
_agg['cobertura']  = _agg['dias_com_dados'] / _agg['n_uteis']
_agg['periodo']    = pd.to_datetime(dict(year=_agg['ano'], month=_agg['mes'], day=1))

# mes corrente (em andamento) e meses com cobertura baixa saem
_mes_corrente = pd.Timestamp.today().normalize().replace(day=1)
_completo = (_agg['cobertura'] >= COBERTURA_MIN) & (_agg['periodo'] < _mes_corrente)
tab_mensal = _agg[_completo].sort_values('periodo').reset_index(drop=True)

# medias por dia util (alvos estaveis)
tab_mensal['media_valor'] = tab_mensal['tot_valor'] / tab_mensal['n_uteis']
tab_mensal['media_qntd']  = tab_mensal['tot_qntd']  / tab_mensal['n_uteis']

print(f'Meses completos: {len(tab_mensal)} | '
      f'{tab_mensal["periodo"].min().date()} -> {tab_mensal["periodo"].max().date()}')
_desc = _agg[~_completo][['ano', 'mes', 'cobertura']]
if len(_desc):
    print('Descartados (parcial/corrente):')
    print(_desc.to_string(index=False))
tab_mensal[['periodo', 'n_uteis', 'tot_valor', 'media_valor', 'tot_qntd', 'media_qntd', 'cobertura']]

## 3 - Tabela supervisionada (lags + calendario) e serie mensal

Para cada alvo (`valor` / `qntd`) montamos:
- **`serie_m`**: serie mensal do total (index mensal `MS`) -> usada pelo SARIMA.
- **`tab_sup`**: tabela supervisionada com features **conhecidas antes do mes t**:
  - `n_uteis` (deterministico do mes t);
  - `lag1/2/3_media` (media por dia util dos meses anteriores);
  - `mm3_media` (media movel de 3 meses da media-util, defasada);
  - `mes_sin`, `mes_cos` (codificacao ciclica do mes -- captura sazonalidade com poucos parametros).

  As primeiras linhas (sem lags) saem da tabela supervisionada.

In [ ]:
FEATS = ['n_uteis', 'lag1_media', 'lag2_media', 'lag3_media', 'mm3_media', 'mes_sin', 'mes_cos']

def serie_mensal(target):
    """Serie mensal do TOTAL (index datetime freq MS). target in {'valor','qntd'}."""
    _col = 'tot_valor' if target == 'valor' else 'tot_qntd'
    s = tab_mensal.set_index('periodo')[_col].astype(float)
    s.index = pd.DatetimeIndex(s.index).to_period('M').to_timestamp()
    return s.asfreq('MS')

def montar_tabela(target):
    """Tabela supervisionada 1-linha-por-mes. Alvo = media por dia util (col 'media').
       Reconstrucao do total = media x n_uteis. Retorna tab_sup (linhas com lags validos)."""
    _mcol = 'media_valor' if target == 'valor' else 'media_qntd'
    _tcol = 'tot_valor'   if target == 'valor' else 'tot_qntd'
    t = tab_mensal.copy()
    t['media'] = t[_mcol].astype(float)
    t['tot']   = t[_tcol].astype(float)
    for _k in (1, 2, 3):
        t[f'lag{_k}_media'] = t['media'].shift(_k)
    t['mm3_media'] = t['media'].shift(1).rolling(3, min_periods=1).mean()   # so passado (shift antes)
    _m = t['periodo'].dt.month
    t['mes_sin'] = np.sin(2 * np.pi * _m / 12.0)
    t['mes_cos'] = np.cos(2 * np.pi * _m / 12.0)
    tab_sup = t.dropna(subset=FEATS).reset_index(drop=True)
    return tab_sup

print('Exemplo VALOR:')
_ex = montar_tabela('valor')
print(f'  serie_m: {len(serie_mensal("valor"))} meses | tab_sup: {len(_ex)} linhas usaveis '
      f'({_ex["periodo"].min().date()} -> {_ex["periodo"].max().date()})')
_ex[['periodo', 'n_uteis', 'media', 'lag1_media', 'lag2_media', 'lag3_media', 'mm3_media', 'tot']]

## 4 - Modelos e avaliacao (janela expansivel, sem vazamento)

Cada mes `t` (a partir do `MIN_TREINO_M`-esimo) e previsto usando **apenas** os meses `< t`.
O modelo e reajustado a cada passo (expanding window). Metrica principal: **MAPE** por mes + agregado.

- **Regressoes** preveem a `media` (por dia util) e reconstroem o total como `media x n_uteis[t]`.
- **Baselines**: `Naive (mes-1)` = total do mes anterior; `Sazonal (mes-12)` = total 12 meses atras;
  `MediaUtil MM3` = media-util dos 3 meses anteriores x n_uteis[t].
- **SARIMA**: ajusta a serie mensal do total e preve 1 passo.

In [ ]:
def _make_ridge():  return make_pipeline(StandardScaler(), Ridge(alpha=1.0))
def _make_linear(): return make_pipeline(StandardScaler(), LinearRegression())
def _make_rf():     return RandomForestRegressor(n_estimators=300, max_depth=3,
                                                 min_samples_leaf=2, random_state=42)

def wf_regressao(tab_sup, make_model, min_treino=MIN_TREINO_M):
    """Walk-forward expansivel: preve a media-util e reconstroi o total (x n_uteis)."""
    rows = []
    for i in range(min_treino, len(tab_sup)):
        tr, te = tab_sup.iloc[:i], tab_sup.iloc[i]
        Xtr, ytr = tr[FEATS].values, tr['media'].values
        m = make_model(); m.fit(Xtr, ytr)
        _media_hat = float(m.predict(te[FEATS].values.reshape(1, -1))[0])
        rows.append({'periodo': te['periodo'],
                     'Previsto':  _media_hat * float(te['n_uteis']),
                     'Realizado': float(te['tot'])})
    return pd.DataFrame(rows)

def wf_baseline(tab_sup, tipo, target, min_treino=MIN_TREINO_M):
    """tipo in {'naive','sazonal','mm3'}. Usa colunas ja defasadas (sem vazamento)."""
    _tcol = 'tot_valor' if target == 'valor' else 'tot_qntd'
    rows = []
    for i in range(min_treino, len(tab_sup)):
        te = tab_sup.iloc[i]
        if tipo == 'naive':
            _prev = float(te['lag1_media']) * float(te['n_uteis'])
        elif tipo == 'mm3':
            _prev = float(te['mm3_media']) * float(te['n_uteis'])
        elif tipo == 'sazonal':
            _p12 = tab_mensal[tab_mensal['periodo'] ==
                              (te['periodo'] - pd.DateOffset(months=12))]
            if not len(_p12):
                continue
            _prev = float(_p12.iloc[0][_tcol])
        else:
            raise ValueError(tipo)
        rows.append({'periodo': te['periodo'], 'Previsto': _prev, 'Realizado': float(te['tot'])})
    return pd.DataFrame(rows)

def wf_sarima(serie_m, tab_sup, min_treino=MIN_TREINO_M):
    """SARIMA 1-passo na serie mensal do total. Sazonal so com >= 24 meses (2 ciclos)."""
    _datas = list(tab_sup['periodo'])
    _seas_on = len(serie_m.dropna()) >= 24
    rows = []
    for _dt in _datas[min_treino:]:
        _key = pd.Timestamp(_dt).to_period('M').to_timestamp()
        if _key not in serie_m.index:
            continue
        _loc = serie_m.index.get_loc(_key)
        _ytr = serie_m.iloc[:_loc].dropna()
        if len(_ytr) < min_treino:
            continue
        try:
            _mod = pm.auto_arima(_ytr, seasonal=_seas_on, m=12 if _seas_on else 1,
                                 stepwise=True, suppress_warnings=True, error_action='ignore',
                                 max_p=2, max_q=2, max_P=1, max_Q=1, max_d=1, max_D=1)
            _yhat = float(np.asarray(_mod.predict(1))[0])
        except Exception:
            continue
        rows.append({'periodo': _dt, 'Previsto': _yhat, 'Realizado': float(serie_m.loc[_key])})
    return pd.DataFrame(rows)

def _rank_linha(det):
    det = det.dropna(subset=['Previsto', 'Realizado'])
    r = pd.to_numeric(det['Realizado'], errors='coerce').astype(float)
    p = pd.to_numeric(det['Previsto'],  errors='coerce').astype(float)
    ape = (p - r).abs() / r.abs().replace(0, np.nan) * 100
    dif = r - p
    ssr = float(((r - p) ** 2).sum()); sst = float(((r - r.mean()) ** 2).sum())
    return {'MAPE Medio (%)': round(float(ape.mean()), 2),
            'RMSPE (%)':      round(float(np.sqrt(np.mean(np.square(ape.dropna())))), 2),
            'Mediana (%)':    round(float(ape.median()), 2),
            'Max (%)':        round(float(ape.max()), 2),
            'R2':             round(1 - ssr / sst, 3) if sst else np.nan,
            'Vies (real-prev)': round(float(dif.mean()), 1),
            'N':              int(len(det))}

print('Funcoes de avaliacao mensal OK')

In [ ]:
def avaliar_mensal(target, unidade='', rodar_tpot=False):
    """Roda baselines + SARIMA + regressoes (+TPOT opcional) por walk-forward mensal.
       Retorna (detalhes: dict nome->DataFrame, ranking: DataFrame com 'Modelo' em COLUNA)."""
    tab_sup = montar_tabela(target)
    serie_m = serie_mensal(target)
    print(f'=== {target.upper()} | tab_sup N={len(tab_sup)} | avaliados a partir do mes '
          f'{MIN_TREINO_M+1} -> {max(0, len(tab_sup)-MIN_TREINO_M)} meses ===')

    det = {}
    det['Naive (mes-1)']   = wf_baseline(tab_sup, 'naive', target)
    det['Sazonal (mes-12)'] = wf_baseline(tab_sup, 'sazonal', target)
    det['MediaUtil MM3']   = wf_baseline(tab_sup, 'mm3', target)
    det['SARIMA']          = wf_sarima(serie_m, tab_sup)
    det['Ridge']           = wf_regressao(tab_sup, _make_ridge)
    det['Linear']          = wf_regressao(tab_sup, _make_linear)
    det['RandomForest']    = wf_regressao(tab_sup, _make_rf)

    if rodar_tpot and _HAS_TPOT and len(tab_sup) - MIN_TREINO_M >= 4:
        det['TPOT'] = wf_tpot(tab_sup)

    det = {k: v for k, v in det.items() if v is not None and not v.empty}
    ranking = (pd.DataFrame({k: _rank_linha(v) for k, v in det.items()}).T
               .sort_values('RMSPE (%)'))
    ranking.index.name = 'Modelo'
    ranking = ranking.reset_index()   # modelo na COLUNA (convencao CLAUDE.md)
    return det, ranking

def plot_mensal(det, ranking, target, unidade=''):
    """1) previsto x realizado de cada modelo ao longo dos meses; 2) barras de MAPE."""
    _real = None
    for v in det.values():
        if len(v):
            _real = v[['periodo', 'Realizado']].drop_duplicates('periodo'); break
    fig, ax = plt.subplots(figsize=(15, 6))
    if _real is not None:
        ax.plot(_real['periodo'], _real['Realizado'], color='black', lw=2.5,
                marker='o', label='Realizado')
    for _nome, _d in det.items():
        if len(_d):
            ax.plot(_d['periodo'], _d['Previsto'], lw=1.4, marker='.', alpha=0.85, label=_nome)
    ax.set_title(f'{target.upper()} - total mensal: previsto x realizado ({unidade})', fontweight='bold')
    ax.set_xlabel('Mes'); ax.set_ylabel(unidade or 'total'); ax.grid(alpha=0.3); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()

    _ord = ranking.sort_values('MAPE Medio (%)')
    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.bar(_ord['Modelo'], _ord['MAPE Medio (%)'], color='#1f77b4')
    ax.set_title(f'{target.upper()} - MAPE medio por modelo (walk-forward mensal)', fontweight='bold')
    ax.set_ylabel('MAPE (%)'); ax.grid(alpha=0.3, axis='y')
    for _i, _v in enumerate(_ord['MAPE Medio (%)']):
        ax.text(_i, _v, f'{_v:.1f}', ha='center', va='bottom', fontsize=8)
    plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

print('Runner e plots OK')

## 5 - AREA VALOR

In [ ]:
det_v, ranking_v = avaliar_mensal('valor', unidade='Valor das Propostas (R$)')
print('\nRanking VALOR (ordenado por RMSPE):')
display(ranking_v)
plot_mensal(det_v, ranking_v, 'valor', unidade='Valor (R$)')

## 6 - AREA QUANTIDADE

In [ ]:
det_q, ranking_q = avaliar_mensal('qntd', unidade='Quantidade de Propostas')
print('\nRanking QUANTIDADE (ordenado por RMSPE):')
display(ranking_q)
plot_mensal(det_q, ranking_q, 'qntd', unidade='Quantidade')

## 7 - TPOT (AutoML) -- exploratorio

> **Aviso de N pequeno.** Com ~10 meses avaliaveis, o TPOT (busca evolutiva de pipelines)
> tende a **overfittar**. Usamos config leve (`TPOT light`, poucas geracoes) e CV interno pequeno.
> A estrategia: o TPOT descobre uma pipeline sobre a tabela supervisionada; essa pipeline e
> entao **reavaliada por walk-forward** (refit a cada mes). Trate o resultado como indicativo,
> nao como veredito -- compare sempre com Ridge/MediaUtil, que sao mais robustos aqui.

In [ ]:
def _descobre_pipeline_tpot(tab_sup, generations=5, population_size=20, cv=3, random_state=42):
    """Roda o TPOT uma vez p/ descobrir uma pipeline (estrutura). Compativel com TPOT >= 1.x
    (API nova: search_space/scorers) e TPOT 0.1x (API antiga: config_dict/verbosity/scoring)."""
    X, y = tab_sup[FEATS].values, tab_sup['media'].values
    _cv = min(cv, max(2, len(tab_sup) // 3))
    try:                                   # ---- API nova (TPOT >= 1.0) ----
        tp = TPOTRegressor(search_space='linear-light',
                           scorers=['neg_mean_absolute_percentage_error'], scorers_weights=[1],
                           cv=_cv, generations=generations, population_size=population_size,
                           n_jobs=1, random_state=random_state, verbose=1)
        tp.fit(X, y)
    except TypeError:                      # ---- API antiga (TPOT 0.1x) ----
        tp = TPOTRegressor(generations=generations, population_size=population_size, cv=_cv,
                           random_state=random_state, verbosity=2, n_jobs=1,
                           config_dict='TPOT light', scoring='neg_mean_absolute_percentage_error')
        tp.fit(X, y)
    return tp.fitted_pipeline_

def wf_tpot(tab_sup, min_treino=MIN_TREINO_M):
    """Descobre a pipeline (uma vez) e a reavalia por walk-forward (refit por mes).
       Com N pequeno a pipeline do TPOT pode conter passos (KNN, CV interno) que quebram
       ao reajustar em poucas linhas -> nesses meses cai no fallback MM3, e o total de
       fallbacks e reportado (sinaliza instabilidade do AutoML nesse N)."""
    from sklearn.base import clone
    _pipe = _descobre_pipeline_tpot(tab_sup)
    print('[TPOT] pipeline descoberta:', _pipe)
    rows, _fb = [], 0
    for i in range(min_treino, len(tab_sup)):
        tr, te = tab_sup.iloc[:i], tab_sup.iloc[i]
        try:
            m = clone(_pipe); m.fit(tr[FEATS].values, tr['media'].values)
            _mh = float(m.predict(te[FEATS].values.reshape(1, -1))[0])
        except Exception:
            _mh = float(te['mm3_media']); _fb += 1          # fallback: media-util MM3
        rows.append({'periodo': te['periodo'],
                     'Previsto': _mh * float(te['n_uteis']), 'Realizado': float(te['tot'])})
    if _fb:
        print(f'[TPOT] {_fb}/{len(rows)} meses no fallback MM3 (pipeline instavel em N pequeno)')
    return pd.DataFrame(rows)

print('TPOT helpers OK | ligue rodar_tpot=True nas celulas abaixo p/ executar')

In [ ]:
# Execucao pesada -- rode manualmente quando quiser (pode demorar).
RODAR_TPOT = False
if RODAR_TPOT and _HAS_TPOT:
    det_v_tp, ranking_v_tp = avaliar_mensal('valor', unidade='Valor (R$)', rodar_tpot=True)
    display(ranking_v_tp); plot_mensal(det_v_tp, ranking_v_tp, 'valor', unidade='Valor (R$)')
    det_q_tp, ranking_q_tp = avaliar_mensal('qntd', unidade='Quantidade', rodar_tpot=True)
    display(ranking_q_tp); plot_mensal(det_q_tp, ranking_q_tp, 'qntd', unidade='Quantidade')
else:
    print('TPOT desligado (RODAR_TPOT=False ou TPOT indisponivel).')

## 8 - Leitura dos resultados

- Compare cada modelo com os **baselines**. Se `Ridge`/`SARIMA` nao baterem `MediaUtil MM3`,
  o sinal preditivo alem de "dias uteis x nivel recente" e fraco -- esperado com N pequeno.
- `Vies (real-prev) > 0` = o modelo **subestima** o total do mes.
- **Proximos passos** quando houver mais historico (>= 24 meses): ligar sazonalidade anual no
  SARIMA (m=12), acrescentar `lag12_media` a tabela supervisionada e reconsiderar o TPOT.
- Exogenas candidatas: nº de dias uteis por dia-da-semana (mês com mais sextas rende diferente),
  feriados prolongados, e o backlog/fila de inicio de mes.